# Notebook 03: a normal classifier or a one-class detector?

**Class 03_01.** If we *have* labelled attacks, why not train an ordinary (two-class) classifier, as for spam?
This notebook answers with three experiments on the KDD99 protocol of notebook 01:

1. **All attack kinds labelled:** logistic regression and a random forest trained on normal rows + the attack pool.
2. **Leave one category out:** the classifier never saw this kind of attack (a *new* attack, the "zero-day" case).
3. **Few labels:** only 10, 30 or 100 labelled attacks.

The one-class detectors of notebook 02 are the reference: they never saw any attack.

In [1]:
import sys

import numpy as np
import polars as pl
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, ".")
import plots_results as pr

import anomlib as an

an.table_config()
sp = an.kdd_split(seed=0)
Xtr, Xva, Xte, y = sp.X("train"), sp.X("val"), sp.X("test"), sp.y_test
Xp, cp = sp.X("pool"), sp.pool["category"].to_numpy()
cats = sp.test["category"]
print("labelled attack pool:", {c: int((cp == c).sum()) for c in an.CATEGORIES})


def supervised(make, keep: np.ndarray) -> dict:
    """Train on normal training rows (label 0) + the pool rows selected by `keep` (label 1); same threshold rule."""
    X = np.vstack([Xtr, Xp[keep]])
    yy = np.r_[np.zeros(len(Xtr)), np.ones(keep.sum())]
    m = make().fit(X, yy)
    s_va, s_te = m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
    return {**an.evaluate(s_te, y, s_va), **an.recall_by_category(s_te, cats, s_va)}


MODELS = {"logistic regression": lambda: LogisticRegression(max_iter=2000),
          "random forest": lambda: RandomForestClassifier(200, n_jobs=-1, random_state=0)}  # fmt: skip

labelled attack pool: {'dos': 54472, 'probe': 2031, 'r2l': 899, 'u2r': 26}


## A. Every kind of attack labelled

In [2]:
rows = [{"model": k, **supervised(f, np.ones(len(cp), bool))} for k, f in MODELS.items()]
for det in [an.GMM(), an.PCARecon()]:
    det.fit(Xtr)
    s_va, s_te = det.score(Xva), det.score(Xte)
    rows.append(
        {"model": det.name + " (one-class)", **an.evaluate(s_te, y, s_va), **an.recall_by_category(s_te, cats, s_va)}
    )
print(pl.DataFrame(rows))
rows_all = rows
pr.clf_all(rows_all)  # slide plots: assets/figures/plots/03/

┌─────────────────────────────────┬───────┬───────┬────────┬───────────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ model                           ┆ AUROC ┆ AUPRC ┆ recall ┆ precision ┆ F1    ┆ FPR   ┆ dos   ┆ probe ┆ r2l   ┆ u2r   │
╞═════════════════════════════════╪═══════╪═══════╪════════╪═══════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╡
│ logistic regression             ┆ 0.980 ┆ 0.902 ┆ 0.951  ┆ 0.633     ┆ 0.760 ┆ 0.010 ┆ 1.000 ┆ 0.980 ┆ 0.920 ┆ 0.769 │
│ random forest                   ┆ 1.000 ┆ 0.998 ┆ 1.000  ┆ 0.628     ┆ 0.772 ┆ 0.011 ┆ 1.000 ┆ 1.000 ┆ 1.000 ┆ 1.000 │
│ GMM (-log-likelihood) (one-cla… ┆ 0.982 ┆ 0.675 ┆ 0.702  ┆ 0.543     ┆ 0.612 ┆ 0.011 ┆ 1.000 ┆ 0.850 ┆ 0.250 ┆ 0.731 │
│ PCA reconstruction (one-class)  ┆ 0.982 ┆ 0.549 ┆ 0.776  ┆ 0.541     ┆ 0.637 ┆ 0.012 ┆ 0.970 ┆ 0.880 ┆ 0.450 ┆ 0.885 │
└─────────────────────────────────┴───────┴───────┴────────┴───────────┴───────┴───────┴───────┴───────┴───────┴───────┘
wrote assets/figures/plots/03/cl

With labels for every kind of attack the random forest is almost perfect (AUPRC near 1): KDD99 is "too easy" for
supervised learning (Tavallaee et al. 2009). Precision stays near 0.63 only because the threshold allows 1% false
alarms; a supervised model can afford a much stricter one.

## B. Leave one category out (a new kind of attack)

In [3]:
rows = []
for name, f in MODELS.items():
    for c in an.CATEGORIES:
        r = supervised(f, cp != c)
        rows.append({"model": name, "unseen": c, "recall on unseen": r[c], "AUPRC": r["AUPRC"]})
loco = pl.DataFrame(rows)
print(loco.pivot(on="unseen", index="model", values="recall on unseen"))
for det in [an.GMM(), an.PCARecon()]:
    det.fit(Xtr)
    rec = an.recall_by_category(det.score(Xte), cats, det.score(Xva))
    print("one-class (never saw any attack):", det.name, rec)
    if det.name.startswith("PCA"):
        pr.loco(loco.to_dicts(), rec)  # slide plot

┌─────────────────────┬───────┬───────┬───────┬───────┐
│ model               ┆ dos   ┆ probe ┆ r2l   ┆ u2r   │
╞═════════════════════╪═══════╪═══════╪═══════╪═══════╡
│ logistic regression ┆ 0.960 ┆ 0.470 ┆ 0.080 ┆ 0.654 │
│ random forest       ┆ 1.000 ┆ 1.000 ┆ 0.360 ┆ 0.962 │
└─────────────────────┴───────┴───────┴───────┴───────┘
one-class (never saw any attack): GMM (-log-likelihood) {'dos': 1.0, 'probe': 0.85, 'r2l': 0.25, 'u2r': 0.7307692307692307}
one-class (never saw any attack): PCA reconstruction {'dos': 0.97, 'probe': 0.88, 'r2l': 0.45, 'u2r': 0.8846153846153846}
wrote assets/figures/plots/03/loco.pdf


**The lesson of the module.** The random forest catches unseen dos, probe and u2r (they look like *some* attack it
knows), but its recall on **unseen r2l falls from 1.00 to 0.36**; logistic regression drops to 0.08. A classifier
learns the boundary between *normal* and *the attacks it saw*; an attack on the normal side of that boundary is
invisible. A one-class detector learns only *normal*: weaker on known attacks, but its recall does not depend on
having seen the attack (PCA: 0.45 on r2l, GMM 0.25, both better than logistic regression's 0.08; r2l is hard for
every method because it *is* close to normal). In practice both run side by side (signatures and classifiers for the known, anomaly detection for the
unknown).

## C. Few labels

In [4]:
rng = np.random.default_rng(0)
rows = []
for n in [10, 30, 100, 300]:
    for rep in range(3):
        keep = np.zeros(len(cp), bool)
        keep[rng.choice(len(cp), n, replace=False)] = True
        rows.append({"labelled attacks": n, "rep": rep, **supervised(MODELS["random forest"], keep)})
few = pl.DataFrame(rows).group_by("labelled attacks").agg(pl.col(c).mean() for c in ["AUPRC", "recall", "r2l", "u2r"])
print(few.sort("labelled attacks"))
pr.fewlabels(rows, float(next(r["AUPRC"] for r in rows_all if r["model"].startswith("GMM"))))  # slide plot

┌──────────────────┬───────┬────────┬───────┬───────┐
│ labelled attacks ┆ AUPRC ┆ recall ┆ r2l   ┆ u2r   │
╞══════════════════╪═══════╪════════╪═══════╪═══════╡
│ 10               ┆ 0.538 ┆ 0.536  ┆ 0.010 ┆ 0.077 │
│ 30               ┆ 0.664 ┆ 0.668  ┆ 0.253 ┆ 0.282 │
│ 100              ┆ 0.802 ┆ 0.823  ┆ 0.563 ┆ 0.590 │
│ 300              ┆ 0.959 ┆ 0.981  ┆ 0.967 ┆ 0.885 │
└──────────────────┴───────┴────────┴───────┴───────┘
wrote assets/figures/plots/03/fewlabels.pdf


A random draw of labels is 95% DoS (that is what the pool holds), so with 10 or 30 labels the forest has seen almost
no r2l or u2r and misses them: with 10 labels its AUPRC (0.54) is below the one-class GMM (0.68), at 30 labels they tie,
from 100 labels on the forest wins. **Semi-supervised** methods
(Deep SAD, notebook 04's references) use both: a one-class model whose few labelled anomalies push the boundary.